In [ ]:
import os, json, random, gc, urllib.request
import numpy as np, torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_kesh"
    print("кэш модели на гугл-диске")
except Exception:
    print("гугл-диск не подключен, кэш локальный")

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN подключен")
except Exception:
    print("HF_TOKEN не задан, качаем анонимно")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BAZA = "https://raw.githubusercontent.com/syeva1/tbank-red-level/main"
for fl in ["data/kid_adult.jsonl", "data/good_bad.jsonl",
           "data/public_test_style.jsonl", "data/public_test_quality.jsonl",
           "metrics/style_clf.pkl",
           "tok/tokenizer.json", "tok/tokenizer_config.json", "tok/vocab.json",
           "tok/merges.txt", "tok/special_tokens_map.json", "tok/added_tokens.json",
           "tok/chat_template.jinja"]:
    os.makedirs(os.path.dirname(fl), exist_ok=True)
    if not os.path.exists(fl):
        urllib.request.urlretrieve(BAZA + "/" + fl, fl)

DAN_DIR = "data"
MTR_DIR = "metrics"
print("файлы на месте:", sorted(os.listdir(DAN_DIR)), sorted(os.listdir(MTR_DIR)))

def grzjl(p):
    with open(p) as f:
        return [json.loads(l) for l in f]

!pip install -q "trl==0.23.0" "transformers==4.57.1" "peft==0.17.1" bitsandbytes accelerate datasets
import trl, transformers, peft
print("trl", trl.__version__, "transformers", transformers.__version__, "peft", peft.__version__)

import torch
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification,
                          AutoTokenizer, BitsAndBytesConfig)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training

MODEL = "Qwen/Qwen3-4B-Instruct-2507"

from huggingface_hub import snapshot_download

def kachml():
    for pop in range(20):
        try:
            return snapshot_download(MODEL, max_workers=8)
        except Exception as osh:
            print("докачка, попытка", pop + 2, type(osh).__name__)
    raise RuntimeError("модель не скачалась")

MPATH = kachml()
print("модель скачана:", MPATH)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained("tok")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

LORTGT = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def lorcfg(tt="CAUSAL_LM"):
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type=tt, target_modules=LORTGT)

def grzbaz():
    m = AutoModelForCausalLM.from_pretrained(
        MPATH, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
    m.config.use_cache = False
    return m

def osvob():
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def slgp(mod, prom, otv):
    pred = tok.apply_chat_template(
        [{"role": "user", "content": prom}],
        add_generation_prompt=True, return_tensors="pt").to(mod.device)
    poln = tok.apply_chat_template(
        [{"role": "user", "content": prom}, {"role": "assistant", "content": otv}],
        return_tensors="pt").to(mod.device)
    metk = poln.clone()
    metk[:, :pred.shape[1]] = -100
    lgts = mod(poln).logits[:, :-1, :]
    metk = metk[:, 1:]
    mask = metk != -100
    ind = metk.clone(); ind[~mask] = 0
    lp = torch.log_softmax(lgts.float(), dim=-1)
    tlp = lp.gather(-1, ind.unsqueeze(-1)).squeeze(-1) * mask
    return (tlp.sum() / mask.sum().clamp(min=1)).item()

def ocpref(mod, tst):
    mod.eval()
    popal = 0
    for t in tst:
        popal += slgp(mod, t["prompt"], t["chosen"]) > slgp(mod, t["prompt"], t["rejected"])
    return popal / len(tst)

def interv(zn, grn):
    for bukva, lo, hi in grn:
        if lo <= zn < hi:
            return bukva
    return "?"

from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ka = grzjl(os.path.join(DAN_DIR, "kid_adult.jsonl"))

sftnab = Dataset.from_list([{
    "prompt":     [{"role": "user", "content": r["prompt"]}],
    "completion": [{"role": "assistant", "content": r["kid"]}],
} for r in ka])

baz = prepare_model_for_kbit_training(grzbaz())
sftarg = SFTConfig(
    output_dir="sft", per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, save_strategy="no", bf16=True, max_length=1024, seed=SEED,
    report_to="none", gradient_checkpointing=True)
sft = SFTTrainer(model=baz, args=sftarg, train_dataset=sftnab,
                 processing_class=tok, peft_config=lorcfg())
sft.train()

from trl import DPOTrainer, DPOConfig

stlnab = Dataset.from_list([{
    "prompt":   [{"role": "user", "content": r["prompt"]}],
    "chosen":   [{"role": "assistant", "content": r["kid"]}],
    "rejected": [{"role": "assistant", "content": r["adult"]}],
} for r in ka])

dpoarg = DPOConfig(
    output_dir="dpo_style", per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=5e-6, beta=0.1, lr_scheduler_type="cosine",
    warmup_ratio=0.05, logging_steps=10, save_strategy="no", bf16=True,
    max_length=1024, max_prompt_length=512, seed=SEED, report_to="none",
    gradient_checkpointing=True)
dpo = DPOTrainer(model=sft.model, args=dpoarg, train_dataset=stlnab, processing_class=tok)
dpo.train()
dpo.save_model("style_adapter")

del sft, dpo, baz; osvob()

from trl import CPOTrainer, CPOConfig

gb = grzjl(os.path.join(DAN_DIR, "good_bad.jsonl"))

baz5 = PeftModel.from_pretrained(grzbaz(), "style_adapter", is_trainable=True)

kachnab = Dataset.from_list([{
    "prompt":   [{"role": "user", "content": r["instruction"]}],
    "chosen":   [{"role": "assistant", "content": r["chosen"]}],
    "rejected": [{"role": "assistant", "content": r["rejected"]}],
} for r in gb])

spoarg = CPOConfig(
    output_dir="simpo", loss_type="simpo", cpo_alpha=0.0, beta=2.0, simpo_gamma=1.0,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=5e-6, lr_scheduler_type="cosine",
    warmup_ratio=0.05, logging_steps=10, save_strategy="no", bf16=True,
    max_length=1024, max_prompt_length=512, seed=SEED, report_to="none",
    gradient_checkpointing=True)
spo = CPOTrainer(model=baz5, args=spoarg, train_dataset=kachnab, processing_class=tok)
spo.train()

tstka = grzjl(os.path.join(DAN_DIR, "public_test_quality.jsonl"))
GRN5 = [("Г", 0.9, 1.0001), ("В", 0.75, 0.9), ("А", 0.6, 0.75), ("Б", 0.0, 0.6)]
acc = ocpref(spo.model, tstka)
print(f"accuracy = {acc:.4f}  ->  {interv(acc, GRN5)}")